# SurviveX Accuracy Validation

Validates all SurviveX models against lifelines on **real datasets** (rossi, lung, gbsg2).
Focus: correctness first, timing second.

In [ ]:
import numpy as np
import pandas as pd
import time
import warnings
warnings.filterwarnings('ignore')

from survivex.models import (
    KaplanMeierEstimator,
    NelsonAalenEstimator,
    CoxPHModel,
    WeibullAFTFitter,
    RandomSurvivalForest,
    GradientBoostingSurvivalAnalysis,
)

import lifelines
from lifelines import KaplanMeierFitter, NelsonAalenFitter, CoxPHFitter
from lifelines import WeibullAFTFitter as LifelinesWeibullAFT
from lifelines.datasets import load_rossi, load_lung, load_gbsg2

print(f"lifelines: {lifelines.__version__}")

results = []

def timed(fn, *args, **kwargs):
    start = time.time()
    result = fn(*args, **kwargs)
    return result, time.time() - start

## Data Preparation

In [ ]:
# Rossi
rossi = load_rossi()
rossi_T = rossi['week'].values.astype(float)
rossi_E = rossi['arrest'].values.astype(float)
rossi_covariates = ['fin', 'age', 'race', 'wexp', 'mar', 'paro', 'prio']
rossi_X = rossi[rossi_covariates].values.astype(float)

# Lung
lung_raw = load_lung()
lung_cols = ['age', 'sex', 'ph.ecog', 'ph.karno', 'pat.karno', 'meal.cal', 'wt.loss']
lung = lung_raw.dropna(subset=['time', 'status'] + lung_cols).copy()
lung_T = lung['time'].values.astype(float)
lung_E = lung['status'].values.astype(float)
lung_X = lung[lung_cols].values.astype(float)

# GBSG2
gbsg2_raw = load_gbsg2()
gbsg2 = gbsg2_raw.copy()
gbsg2['horTh_num'] = (gbsg2['horTh'] == 'yes').astype(float)
gbsg2['menostat_num'] = (gbsg2['menostat'] == 'Post').astype(float)
tgrade_map = {'I': 1, 'II': 2, 'III': 3}
gbsg2['tgrade_num'] = gbsg2['tgrade'].map(tgrade_map).astype(float)
gbsg2_covariates = ['horTh_num', 'age', 'menostat_num', 'tsize', 'tgrade_num', 'pnodes', 'progrec', 'estrec']
gbsg2_T = gbsg2['time'].values.astype(float)
gbsg2_E = gbsg2['cens'].values.astype(float)
gbsg2_X = gbsg2[gbsg2_covariates].values.astype(float)

print("Dataset summaries:")
print(f"  rossi:  n={len(rossi_T):>4}, events={int(rossi_E.sum()):>4}, features={rossi_X.shape[1]}")
print(f"  lung:   n={len(lung_T):>4}, events={int(lung_E.sum()):>4}, features={lung_X.shape[1]}")
print(f"  gbsg2:  n={len(gbsg2_T):>4}, events={int(gbsg2_E.sum()):>4}, features={gbsg2_X.shape[1]}")

## Kaplan-Meier Validation (rossi)

In [ ]:
km_sx = KaplanMeierEstimator()
_, t_sx = timed(km_sx.fit, rossi_T, rossi_E)

km_ll = KaplanMeierFitter()
_, t_ll = timed(km_ll.fit, rossi_T, rossi_E)

test_times = km_ll.survival_function_.index.values
sx_vals = km_sx.survival_function_at_times(test_times).numpy()
ll_vals = km_ll.survival_function_at_times(test_times).values

km_max_diff = np.max(np.abs(sx_vals - ll_vals))

print("Kaplan-Meier Validation (rossi, n=432)")
print(f"  Max |S(t) difference|: {km_max_diff:.2e}")
print(f"  Timing: SurviveX={t_sx:.5f}s, lifelines={t_ll:.5f}s")
print(f"  PASS" if km_max_diff < 1e-6 else f"  FAIL")

results.append({
    'Model': 'Kaplan-Meier', 'Dataset': 'rossi', 'Metric': 'max|S(t) diff|',
    'Difference': km_max_diff, 'Tolerance': 1e-6,
    'Pass': km_max_diff < 1e-6, 'SurviveX_time': t_sx, 'lifelines_time': t_ll
})

## Nelson-Aalen Validation (rossi)

In [ ]:
na_sx = NelsonAalenEstimator()
_, t_sx = timed(na_sx.fit, rossi_T, rossi_E)

na_ll = NelsonAalenFitter()
_, t_ll = timed(na_ll.fit, rossi_T, rossi_E)

test_times = na_ll.cumulative_hazard_.index.values
sx_vals = na_sx.cumulative_hazard_at_times(test_times).numpy()
ll_vals = na_ll.cumulative_hazard_at_times(test_times).values

na_max_diff = np.max(np.abs(sx_vals - ll_vals))

print("Nelson-Aalen Validation (rossi, n=432)")
print(f"  Max |H(t) difference|: {na_max_diff:.2e}")
print(f"  Timing: SurviveX={t_sx:.5f}s, lifelines={t_ll:.5f}s")
print(f"  PASS" if na_max_diff < 1e-3 else f"  FAIL")

results.append({
    'Model': 'Nelson-Aalen', 'Dataset': 'rossi', 'Metric': 'max|H(t) diff|',
    'Difference': na_max_diff, 'Tolerance': 1e-3,
    'Pass': na_max_diff < 1e-3, 'SurviveX_time': t_sx, 'lifelines_time': t_ll
})

## Cox PH Validation - Breslow (rossi)

In [ ]:
cox_sx_b = CoxPHModel(device='cpu', tie_method='breslow')
_, t_sx = timed(cox_sx_b.fit, rossi_X, rossi_T, rossi_E)

df_rossi_cox = rossi[rossi_covariates + ['week', 'arrest']].copy()
cox_ll_b = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
_, t_ll = timed(cox_ll_b.fit, df_rossi_cox, duration_col='week', event_col='arrest')

# R reference: coxph(Surv(week, arrest) ~ ., data=rossi, method="breslow")
r_breslow = np.array([-0.37902188743, -0.05724592504, 0.31412976691,
                      -0.15111460006, -0.43278257376, -0.08498283527, 0.09111154209])

coef_diff_b = np.max(np.abs(cox_sx_b.coefficients_ - r_breslow))
breslow_vs_efron = np.max(np.abs(cox_sx_b.coefficients_ - cox_ll_b.params_.values))

print("Cox PH Validation - Breslow (rossi, n=432, p=7)")
print(f"  Max |coef diff| vs R Breslow:  {coef_diff_b:.2e}")
print(f"  Breslow vs Efron diff:         {breslow_vs_efron:.2e}  (expected ~1e-3 with ties)")
print(f"  Timing: SurviveX={t_sx:.4f}s, lifelines(Efron)={t_ll:.4f}s")
print(f"  PASS" if coef_diff_b < 1e-4 else f"  FAIL")

print("\n  Coefficient Comparison (vs R Breslow):")
print(f"  {'Covariate':<10} {'SurviveX':>12} {'R Breslow':>12} {'diff':>10}")
for i, name in enumerate(rossi_covariates):
    print(f"  {name:<10} {cox_sx_b.coefficients_[i]:>12.8f} {r_breslow[i]:>12.8f} {abs(cox_sx_b.coefficients_[i]-r_breslow[i]):>10.2e}")

results.append({
    'Model': 'Cox PH (Breslow)', 'Dataset': 'rossi', 'Metric': 'max|coef diff| vs R',
    'Difference': coef_diff_b, 'Tolerance': 1e-4,
    'Pass': coef_diff_b < 1e-4, 'SurviveX_time': t_sx, 'lifelines_time': t_ll
})

## Cox PH Validation - Efron (rossi)

In [ ]:
cox_sx_e = CoxPHModel(device='cpu', tie_method='efron')
_, t_sx = timed(cox_sx_e.fit, rossi_X, rossi_T, rossi_E)

cox_ll_e = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
_, t_ll = timed(cox_ll_e.fit, df_rossi_cox, duration_col='week', event_col='arrest')

coef_diff_e = np.max(np.abs(cox_sx_e.coefficients_ - cox_ll_e.params_.values))
se_diff_e = np.max(np.abs(cox_sx_e.standard_errors_ - cox_ll_e.summary['se(coef)'].values))
hr_diff_e = np.max(np.abs(np.exp(cox_sx_e.coefficients_) - cox_ll_e.summary['exp(coef)'].values))
ci_sx_e = cox_sx_e.concordance_index_
ci_ll_e = cox_ll_e.concordance_index_

from scipy.stats import norm
z_sx = cox_sx_e.coefficients_ / cox_sx_e.standard_errors_
p_sx = 2 * (1 - norm.cdf(np.abs(z_sx)))
p_ll = cox_ll_e.summary['p'].values
p_diff_e = np.max(np.abs(p_sx - p_ll))

print("Cox PH Validation - Efron (rossi, n=432, p=7)")
print(f"  Max |coef diff|:    {coef_diff_e:.2e}")
print(f"  Max |SE diff|:      {se_diff_e:.2e}")
print(f"  Max |HR diff|:      {hr_diff_e:.2e}")
print(f"  Max |p-value diff|: {p_diff_e:.2e}")
print(f"  C-index: SurviveX={ci_sx_e:.6f}, lifelines={ci_ll_e:.6f}, diff={abs(ci_sx_e-ci_ll_e):.2e}")
print(f"  Timing: SurviveX={t_sx:.4f}s, lifelines={t_ll:.4f}s")
print(f"  PASS" if coef_diff_e < 1e-4 else f"  FAIL")

print("\n  Coefficient Comparison:")
print(f"  {'Covariate':<10} {'SurviveX':>10} {'lifelines':>10} {'diff':>10}")
for i, name in enumerate(rossi_covariates):
    print(f"  {name:<10} {cox_sx_e.coefficients_[i]:>10.6f} {cox_ll_e.params_.values[i]:>10.6f} {abs(cox_sx_e.coefficients_[i]-cox_ll_e.params_.values[i]):>10.2e}")

results.append({
    'Model': 'Cox PH (Efron)', 'Dataset': 'rossi', 'Metric': 'max|coef diff|',
    'Difference': coef_diff_e, 'Tolerance': 1e-4,
    'Pass': coef_diff_e < 1e-4, 'SurviveX_time': t_sx, 'lifelines_time': t_ll
})

## Weibull AFT Validation (rossi)

In [ ]:
wb_sx = WeibullAFTFitter()
_, t_sx = timed(wb_sx.fit, rossi_X, rossi_T, rossi_E)

df_wb = rossi[rossi_covariates + ['week', 'arrest']].copy()
wb_ll = LifelinesWeibullAFT(penalizer=0.0)
_, t_ll = timed(wb_ll.fit, df_wb, duration_col='week', event_col='arrest')

# Compare predicted survival at mean covariates
X_mean = rossi_X.mean(axis=0).reshape(1, -1)
test_times = np.array([10, 20, 30, 40, 50, 60, 70, 80])

sx_surv = wb_sx.predict_survival_function(X_mean, times=test_times)

df_mean = pd.DataFrame(X_mean, columns=rossi_covariates)
ll_surv = wb_ll.predict_survival_function(df_mean, times=test_times).values.flatten()

surv_diff = np.max(np.abs(sx_surv - ll_surv))
sx_rho = wb_sx.rho_

print("Weibull AFT Validation (rossi, n=432, p=7)")
print(f"  SurviveX shape (rho): {sx_rho:.4f}")
print(f"  Max |S(t) diff| at mean X: {surv_diff:.2e}")
print(f"  Timing: SurviveX={t_sx:.5f}s, lifelines={t_ll:.5f}s")
print(f"  Speedup: {t_ll/t_sx:.1f}x")
print(f"  PASS" if surv_diff < 0.05 else f"  FAIL")

print("\n  Predicted S(t) at mean covariates:")
print(f"  {'t':<6} {'SurviveX':>10} {'lifelines':>10} {'diff':>10}")
for i, t in enumerate(test_times):
    print(f"  {t:<6.0f} {sx_surv[i]:>10.6f} {ll_surv[i]:>10.6f} {abs(sx_surv[i]-ll_surv[i]):>10.2e}")

results.append({
    'Model': 'Weibull AFT', 'Dataset': 'rossi', 'Metric': 'max|S(t) diff|',
    'Difference': surv_diff, 'Tolerance': 0.05,
    'Pass': surv_diff < 0.05, 'SurviveX_time': t_sx, 'lifelines_time': t_ll
})

## Random Survival Forest Validation (rossi)

In [ ]:
# Train/test split
np.random.seed(42)
n = len(rossi_T)
idx = np.random.permutation(n)
train_idx, test_idx = idx[:int(0.7*n)], idx[int(0.7*n):]

X_train, X_test = rossi_X[train_idx], rossi_X[test_idx]
T_train, T_test = rossi_T[train_idx], rossi_T[test_idx]
E_train, E_test = rossi_E[train_idx], rossi_E[test_idx]

rsf = RandomSurvivalForest(n_estimators=100, max_depth=5, random_state=42)
_, t_sx = timed(rsf.fit, X_train, T_train, E_train)

ci_rsf = rsf.score(X_test, T_test, E_test)

# Cox PH baseline for reference
df_train = pd.DataFrame(X_train, columns=rossi_covariates)
df_train['T'], df_train['E'] = T_train, E_train
df_test = pd.DataFrame(X_test, columns=rossi_covariates)
df_test['T'], df_test['E'] = T_test, E_test

cox_baseline = CoxPHFitter(penalizer=0.0)
cox_baseline.fit(df_train, duration_col='T', event_col='E')
ci_cox_baseline = cox_baseline.score(df_test, scoring_method='concordance_index')

print("Random Survival Forest Validation (rossi, n=432, 70/30 split)")
print(f"  RSF C-index (test):      {ci_rsf:.4f}")
print(f"  Cox PH C-index (test):   {ci_cox_baseline:.4f}  (baseline reference)")
print(f"  RSF OOB C-index:         {rsf.oob_score_:.4f}")
print(f"  Timing: RSF fit={t_sx:.4f}s")
print(f"  PASS" if ci_rsf > 0.55 else f"  FAIL (C-index too low)")

results.append({
    'Model': 'Random Survival Forest', 'Dataset': 'rossi', 'Metric': 'C-index (test)',
    'Difference': ci_rsf, 'Tolerance': 0.55,
    'Pass': ci_rsf > 0.55, 'SurviveX_time': t_sx, 'lifelines_time': None
})

## Gradient Boosting Validation (rossi)

In [ ]:
gb = GradientBoostingSurvivalAnalysis(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)
_, t_sx = timed(gb.fit, X_train, T_train, E_train)

ci_gb = gb.score(X_test, T_test, E_test)

print("Gradient Boosting Validation (rossi, n=432, 70/30 split)")
print(f"  GB C-index (test):       {ci_gb:.4f}")
print(f"  Cox PH C-index (test):   {ci_cox_baseline:.4f}  (baseline reference)")
print(f"  Timing: GB fit={t_sx:.4f}s")
print(f"  PASS" if ci_gb > 0.55 else f"  FAIL (C-index too low)")

results.append({
    'Model': 'Gradient Boosting', 'Dataset': 'rossi', 'Metric': 'C-index (test)',
    'Difference': ci_gb, 'Tolerance': 0.55,
    'Pass': ci_gb > 0.55, 'SurviveX_time': t_sx, 'lifelines_time': None
})

## Cross-Dataset Validation: Cox PH on gbsg2

In [ ]:
cox_sx_g = CoxPHModel(device='cpu', tie_method='efron')
_, t_sx = timed(cox_sx_g.fit, gbsg2_X, gbsg2_T, gbsg2_E)

df_gbsg2_cox = gbsg2[gbsg2_covariates + ['time', 'cens']].copy()
cox_ll_g = CoxPHFitter(penalizer=0.0, l1_ratio=0.0)
_, t_ll = timed(cox_ll_g.fit, df_gbsg2_cox, duration_col='time', event_col='cens')

coef_diff_g = np.max(np.abs(cox_sx_g.coefficients_ - cox_ll_g.params_.values))
se_diff_g = np.max(np.abs(cox_sx_g.standard_errors_ - cox_ll_g.summary['se(coef)'].values))
ci_sx_g = cox_sx_g.concordance_index_
ci_ll_g = cox_ll_g.concordance_index_

print("Cox PH Cross-Dataset Validation - Efron (gbsg2, n=686, p=8)")
print(f"  Max |coef diff|:    {coef_diff_g:.2e}")
print(f"  Max |SE diff|:      {se_diff_g:.2e}")
print(f"  C-index: SurviveX={ci_sx_g:.6f}, lifelines={ci_ll_g:.6f}, diff={abs(ci_sx_g-ci_ll_g):.2e}")
print(f"  Timing: SurviveX={t_sx:.4f}s, lifelines={t_ll:.4f}s")
print(f"  PASS" if coef_diff_g < 1e-4 else f"  FAIL")

print("\n  Coefficient Comparison:")
print(f"  {'Covariate':<14} {'SurviveX':>10} {'lifelines':>10} {'diff':>10}")
for i, name in enumerate(gbsg2_covariates):
    print(f"  {name:<14} {cox_sx_g.coefficients_[i]:>10.6f} {cox_ll_g.params_.values[i]:>10.6f} {abs(cox_sx_g.coefficients_[i]-cox_ll_g.params_.values[i]):>10.2e}")

results.append({
    'Model': 'Cox PH (Efron)', 'Dataset': 'gbsg2', 'Metric': 'max|coef diff|',
    'Difference': coef_diff_g, 'Tolerance': 1e-4,
    'Pass': coef_diff_g < 1e-4, 'SurviveX_time': t_sx, 'lifelines_time': t_ll
})

## Results Summary

In [ ]:
print("=" * 90)
print("ACCURACY VALIDATION SUMMARY")
print("=" * 90)

df_results = pd.DataFrame(results)
print(f"\n{'Model':<24} {'Dataset':<8} {'Metric':<20} {'Value':<12} {'Tol/Thresh':<12} {'Status'}")
print("-" * 90)
for _, row in df_results.iterrows():
    status = 'PASS' if row['Pass'] else 'FAIL'
    print(f"{row['Model']:<24} {row['Dataset']:<8} {row['Metric']:<20} {row['Difference']:<12.2e} {row['Tolerance']:<12.2e} {status}")

n_pass = df_results['Pass'].sum()
n_total = len(df_results)
print(f"\nOverall: {n_pass}/{n_total} tests passed")

print("\n" + "=" * 90)
print("TIMING COMPARISON")
print("=" * 90)
print(f"\n{'Model':<24} {'Dataset':<8} {'SurviveX (s)':<14} {'lifelines (s)':<14} {'Speedup'}")
print("-" * 90)
for _, row in df_results.iterrows():
    t_sx = row['SurviveX_time']
    t_ll = row['lifelines_time']
    if t_ll is not None and t_ll > 0:
        speedup = f"{t_ll/t_sx:.1f}x"
    else:
        speedup = "N/A"
    t_ll_str = f"{t_ll:.5f}" if t_ll is not None else "N/A"
    print(f"{row['Model']:<24} {row['Dataset']:<8} {t_sx:<14.5f} {t_ll_str:<14} {speedup}")

print("\n" + "=" * 90)
if n_pass == n_total:
    print("ALL TESTS PASSED")
else:
    print(f"{n_total - n_pass} test(s) failed.")
print("=" * 90)